# 06. K-Nearest Neighbors (KNN) Regression

**Objectif** : Prédire les prix immobiliers avec KNN optimisé

In [1]:
# Importer les bibliothèques
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Modélisation KNN
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.preprocessing import StandardScaler
import time

%matplotlib inline

print("📚 Bibliothèques importées avec succès !")
print("🚀 KNN Regression prêt")

📚 Bibliothèques importées avec succès !
🚀 KNN Regression prêt


In [2]:
# Charger et préparer les données
df = pd.read_csv("../data/real_estate_processed.csv")
print(f"Données originales: {df.shape}")

# Nettoyage
df_clean = df[(df['price'] >= 50) & (df['price'] <= 10000000)].copy()
print(f"Données après nettoyage: {df_clean.shape}")
print(f"Supprimées: {len(df) - len(df_clean)} annonces")

df = df_clean

print(f"\nStatistiques des prix:")
print(f"Prix moyen: {df['price'].mean():.0f} DT")
print(f"Prix médian: {df['price'].median():.0f} DT")
print(f"Prix min: {df['price'].min():.0f} DT")
print(f"Prix max: {df['price'].max():.0f} DT")
print(f"Écart-type: {df['price'].std():.0f} DT")

df.head(3)

Données originales: (5653, 12)
Données après nettoyage: (5601, 12)
Supprimées: 52 annonces

Statistiques des prix:
Prix moyen: 176410 DT
Prix médian: 3500 DT
Prix min: 55 DT
Prix max: 6300000 DT
Écart-type: 300082 DT


,title,price_text,price,category,city,location,type_transaction,rooms,post_time,post_date,post_month,post_year
0,À louer – Bureaux neufs S+1 et S+2 à Monastir ...,650 DT,650,0,Monastir,Monastir,0,1,2/4/26 12:37,02-04-26,2,2026
1,S+1 haut standing pour la saison universitaire,850 DT,850,0,Monastir,Monastir,0,1,8/30/25 10:49,08-30-25,8,2025
2,à vendre s+3 haut standing directement au prom...,350000 DT,350000,1,Monastir,Bekalta,1,3,7/30/25 12:45,07-30-25,7,2025


In [3]:
# Préparation des données pour KNN
print("=== PRÉPARATION DES DONNÉES POUR KNN ===")

# Features numériques
numeric_columns = df.select_dtypes(include=[np.number]).columns.tolist()
X = df[numeric_columns].drop('price', axis=1)
y = df['price']

print(f"Features de base: {X.shape[1]}")
print(f"Features: {X.columns.tolist()}")

# Division des données
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Données divisées: {X_train.shape[0]} entraînement, {X_test.shape[0]} test")

# Standardisation (très important pour KNN)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"\n✅ Données standardisées")
print(f"   Moyenne avant scaling: {X_train.mean().mean():.2f}")
print(f"   Moyenne après scaling: {X_train_scaled.mean():.6f}")
print(f"   Std avant scaling: {X_train.std().mean():.2f}")
print(f"   Std après scaling: {X_train_scaled.std():.6f}")

=== PRÉPARATION DES DONNÉES POUR KNN ===
Features de base: 4
Features: ['category', 'type_transaction', 'post_month', 'post_year']
Données divisées: 4480 entraînement, 1121 test

✅ Données standardisées
   Moyenne avant scaling: 508.19
   Moyenne après scaling: -0.000000
   Std avant scaling: 1.49
   Std après scaling: 1.000000


In [4]:
# KNN - Version simple
print("=== KNN - VERSION SIMPLE ===")

# Créer et entraîner le modèle
start_time = time.time()

knn_simple = KNeighborsRegressor(
    n_neighbors=5,
    weights='uniform',
    algorithm='auto',
    metric='euclidean'
)

knn_simple.fit(X_train_scaled, y_train)
training_time = time.time() - start_time

print(f"⏱️ Temps d'entraînement: {training_time:.2f} secondes")
print(f"👥 Nombre de voisins: {knn_simple.n_neighbors}")
print(f"⚖️ Pondération: {knn_simple.weights}")
print(f"📏 Distance: {knn_simple.metric}")
print(f"✅ Modèle entraîné !")

=== KNN - VERSION SIMPLE ===
⏱️ Temps d'entraînement: 0.01 secondes
👥 Nombre de voisins: 5
⚖️ Pondération: uniform
📏 Distance: euclidean
✅ Modèle entraîné !


In [5]:
# Évaluation du modèle simple
print("=== ÉVALUATION MODÈLE SIMPLE ===")

# Prédictions
y_train_pred = knn_simple.predict(X_train_scaled)
y_test_pred = knn_simple.predict(X_test_scaled)

# Métriques
train_r2 = r2_score(y_train, y_train_pred)
test_r2 = r2_score(y_test, y_test_pred)
train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
train_mae = mean_absolute_error(y_train, y_train_pred)
test_mae = mean_absolute_error(y_test, y_test_pred)

print(f"📊 PERFORMANCE KNN:")
print(f"   R² Train: {train_r2:.4f}")
print(f"   R² Test:  {test_r2:.4f}")
print(f"   RMSE Train: {train_rmse:,.0f} DT")
print(f"   RMSE Test:  {test_rmse:,.0f} DT")
print(f"   MAE Train:  {train_mae:,.0f} DT")
print(f"   MAE Test:   {test_mae:,.0f} DT")

# Analyse de l'overfitting
overfitting_r2 = train_r2 - test_r2
print(f"\n🔍 ANALYSE OVERFITTING:")
print(f"   Différence R² (Train-Test): {overfitting_r2:.4f}")

if overfitting_r2 > 0.1:
    print(f"   ⚠️ Overfitting détecté !")
elif overfitting_r2 > 0.05:
    print(f"   🟡 Léger overfitting")
else:
    print(f"   ✅ Bon équilibre")

# Qualité du modèle
if test_r2 > 0.7:
    quality = "🌟 Excellente"
elif test_r2 > 0.5:
    quality = "✅ Bonne"
elif test_r2 > 0.3:
    quality = "⚠️ Moyenne"
else:
    quality = "📉 Faible"

print(f"\n🏆 QUALITÉ: {quality}")

=== ÉVALUATION MODÈLE SIMPLE ===
📊 PERFORMANCE KNN:
   R² Train: 0.4087
   R² Test:  0.4109
   RMSE Train: 235,847 DT
   RMSE Test:  208,299 DT
   MAE Train:  98,903 DT
   MAE Test:   94,777 DT

🔍 ANALYSE OVERFITTING:
   Différence R² (Train-Test): -0.0023
   ✅ Bon équilibre

🏆 QUALITÉ: ⚠️ Moyenne


📊 PERFORMANCE KNN:
   R² Train: 0.4087
   R² Test:  0.4109
   RMSE Train: 235,847 DT
   RMSE Test:  208,299 DT
   MAE Train:  98,903 DT
   MAE Test:   94,777 DT

🔍 ANALYSE OVERFITTING:
   Différence R² (Train-Test): -0.0023
   ✅ Bon équilibre

🏆 QUALITÉ: ⚠️ Moyenne


In [6]:
# Sauvegarde du modèle KNN
print("=== SAUVEGARDE DU MODÈLE KNN ===")

import joblib
import os

# Créer le dossier models s'il n'existe pas
if not os.path.exists('../models'):
    os.makedirs('../models')

# Sauvegarder le meilleur modèle
joblib.dump(knn_simple, '../models/knn_best_model.pkl')

# Sauvegarder le scaler
joblib.dump(scaler, '../models/knn_scaler.pkl')

# Sauvegarder les résultats
knn_results = {
    'model_name': 'KNN',
    'version': 'Simple',
    'r2_train': train_r2,
    'r2_test': test_r2,
    'rmse_train': train_rmse,
    'rmse_test': test_rmse,
    'mae_train': train_mae,
    'mae_test': test_mae,
    'training_time': training_time,
    'n_features': X.shape[1],
    'feature_columns': list(X.columns),
    'best_k': 5,
    'scaler_name': 'StandardScaler',
    'quality': quality
}

joblib.dump(knn_results, '../models/knn_results.pkl')

print(f"   ✅ Modèle sauvegardé: ../models/knn_best_model.pkl")
print(f"   ✅ Scaler sauvegardé: ../models/knn_scaler.pkl")
print(f"   ✅ Résultats sauvegardés: ../models/knn_results.pkl")
print(f"\n🎉 KNN Regression terminé avec succès !")

=== SAUVEGARDE DU MODÈLE KNN ===
   ✅ Modèle sauvegardé: ../models/knn_best_model.pkl
   ✅ Scaler sauvegardé: ../models/knn_scaler.pkl
   ✅ Résultats sauvegardés: ../models/knn_results.pkl

🎉 KNN Regression terminé avec succès !
